In [ ]:
!pip install pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 7.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
from  google.colab import drive
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import MiniBatchKMeans

In [ ]:
uploaded = drive.mount('/content/drive')
finaccess = '/content/drive/MyDrive/2024_Finaccess_Publicdata.dta'
if finaccess:
    data = pd.read_stata(finaccess, convert_categoricals=False)
    print("Succeess")
else:
    print("Error reading Finaccess")

Mounted at /content/drive
Succeess


In [ ]:
df = pd.DataFrame(data)
df.head()


,interview__key,interview__id,county,A07,A08,A10,Interview_Status,A9i,Z1,A13,...,sacco_redress,mobilemoney_redress,mobilebank_redress,not_registered_mmoney_24,using_someone_acc,insurance_including_NHIF_use,All_Insurance_excluding_NHIF_use,PWD,Latitude,Longitude
0,21-61-43-44,00005e63e7424aa19e63adb071955233,7,225,2,1,1,1,1,1,...,NaN,NaN,NaN,0,NaN,3,3,0,-0.435423,39.636586
1,87-78-54-60,000718d26f5942998522033a66594b4c,7,223,2,3,1,1,1,1,...,NaN,NaN,NaN,0,NaN,3,3,0,0.058794,40.305006
2,32-71-19-84,0008721681d2468aa1ef456bba4475dc,40,1514,2,3,1,1,1,2,...,NaN,NaN,NaN,0,NaN,3,3,0,0.636836,34.277390
3,48-18-78-99,00099e961d49470585c1a7f49fa3b7fb,22,796,2,3,1,1,1,1,...,NaN,NaN,NaN,0,NaN,3,3,0,-1.251917,36.719076
4,50-82-98-89,000bb72e959b4dd48443da902c01e920,21,763,2,16,1,1,1,2,...,NaN,NaN,NaN,0,1.0,3,3,0,-0.795820,37.131085


In [ ]:
df.tail()

,interview__key,interview__id,county,A07,A08,A10,Interview_Status,A9i,Z1,A13,...,sacco_redress,mobilemoney_redress,mobilebank_redress,not_registered_mmoney_24,using_someone_acc,insurance_including_NHIF_use,All_Insurance_excluding_NHIF_use,PWD,Latitude,Longitude
20866,96-94-76-93,ffefe104529e439fb916d956d10a9dc0,8,1847,1,1,1,1,1,1,...,NaN,NaN,NaN,0,NaN,3,3,0,3.328605,39.788873
20867,40-55-72-03,fff39eec827f4ddc90e73a0792092bf9,47,1815,2,25,1,1,1,2,...,NaN,NaN,NaN,0,NaN,2,3,0,-1.263889,36.853431
20868,10-75-82-28,fff70af11a2e49649feae012317ce7d5,14,443,1,1,1,1,1,1,...,NaN,NaN,NaN,0,NaN,2,3,0,-0.521351,37.568370
20869,77-66-40-97,fff8b3b628df4b67889e26af8df90ef7,6,213,2,1,1,1,1,2,...,NaN,NaN,NaN,0,NaN,3,3,0,-3.378276,38.564794
20870,09-26-74-74,fffe9b46c1014de9be537c63c210014c,45,1706,1,1,1,1,1,1,...,NaN,NaN,NaN,1,0.0,3,3,1,-0.619969,34.784302


In [ ]:
#Data cleaning
shape= df.shape[0]
print(shape)
raw_count_before = len(df)
df.drop_duplicates(inplace=True)
raw_count_after = len(df)
print(f"Duplicates removed: {raw_count_before - raw_count_after}")
df.isna().sum()




20871
Duplicates removed: 0


,0
interview__key,0
interview__id,0
county,0
A07,0
A08,0
...,...
insurance_including_NHIF_use,0
All_Insurance_excluding_NHIF_use,0
PWD,0
Latitude,0


In [ ]:
empty_rows_count = df.isna().all(axis=1).sum()
print(empty_rows_count)

0


In [ ]:
df.isna().sum().sum()


np.int64(47772168)

In [ ]:
missing_percentage = (df.isna().sum()/len(df))* 100
missing_df = pd.DataFrame({'Total Missing': df.isna().sum(), 'Percentage':missing_percentage})
print(missing_df[missing_df['Total Missing']>0].sort_values(by='Percentage', ascending=False).head(20))

         Total Missing  Percentage
J3__158          20871       100.0
J3__117          20871       100.0
J3__116          20871       100.0
J3__157          20871       100.0
J3__156          20871       100.0
J3__155          20871       100.0
J3__154          20871       100.0
J3__153          20871       100.0
J3__152          20871       100.0
J3__151          20871       100.0
J3__150          20871       100.0
J3__148          20871       100.0
J3__170          20871       100.0
J3__169          20871       100.0
J3__168          20871       100.0
J3__167          20871       100.0
J3__166          20871       100.0
J3__165          20871       100.0
J3__164          20871       100.0
J3__163          20871       100.0


In [ ]:
empty = df.columns[df.isna().mean()==1.0]
print(f"Dropping {len(empty)} completely empyty")
df.drop(columns=empty, inplace=True)

Dropping 674 completely empyty


In [ ]:
remaining = df.isna().sum()
remaining = remaining[remaining>0].sort_values(ascending=False)
print(remaining.head(20))
print(remaining.sum())

H7i_1      20870
H7ii_1     20870
H6_1       20870
G3__3      20870
H7i_19     20870
H6_19      20870
H7ii_19    20870
H7i_21     20870
H6_21      20870
H7ii_21    20870
H7ii_39    20870
H7i_13     20870
H6_11      20870
H7i_11     20870
H6_13      20870
H7ii_11    20870
H7ii_13    20870
H6_6       20870
H7ii_6     20870
H7i_6      20870
dtype: int64
33705114


In [ ]:
ml_threshold =0.90
initial_count = df.shape[1]
df_ml_cleaned = df.loc[:, df.isna().mean() < ml_threshold]
dropped = initial_count - df_ml_cleaned.shape[1]
print(f"Removed {dropped} sparse/skipped columns. New DataFrame shape: {df_ml_cleaned.shape}")

Removed 951 sparse/skipped columns. New DataFrame shape: (20871, 2191)


In [ ]:
id_cols =['interview__key', 'interview__id']
df_ids = df_ml_cleaned[[col for col in id_cols if col in df_ml_cleaned.columns]]
df_features = df_ml_cleaned.drop(columns=id_cols, errors='ignore')
non_numeric = df_features.select_dtypes(include=['object', 'category']).columns
cardinality = df_features[non_numeric].nunique().sort_values(ascending=False)
print(cardinality.head(20))

# Drop anything with too many categories — these are usually free-text
# or overly granular codes, not useful for one-hot encoding anyway
max_cardinality = 20
high_card = cardinality[cardinality > max_cardinality].index.tolist()
print(f"Dropping {len(high_card)} high-cardinality columns")
df_features = df_features.drop(columns=high_card)

non_numeric = df_features.select_dtypes(include=['object', 'category']).columns
print(f"One-hot encoding {len(non_numeric)} remaining columns")
df_features = pd.get_dummies(df_features, columns=non_numeric, drop_first=True)

# force dummy columns to compact dtype instead of letting pandas upcast
bool_cols = df_features.select_dtypes(include=['bool', 'uint8']).columns
df_features[bool_cols] = df_features[bool_cols].astype(np.uint8)

import gc
gc.collect()

if df_features.isna().sum().sum() > 0:
    df_features = df_features.fillna(df_features.median(numeric_only=True))

EndofInt_Time        20759
B3F1                  4933
P1_2i                 2325
Q7ai                   563
S4_Other_Specify       406
Q14i                   308
D3i                    281
U2_Other_Specify       238
I1a_Other_Specify      233
I1b_Other_Specify      218
Q8ai                   217
R3Bi                   204
P5i                    197
R1Bi                   190
A15i                   170
J3i                    156
J2_Others_Specify      155
N2i_Other_Specify      140
T3i                    133
Q24i                   132
dtype: int64
Dropping 97 high-cardinality columns
One-hot encoding 195 remaining columns


In [ ]:
print("before dummies")
df_features = pd.get_dummies(df_features, columns=non_numeric, drop_first=True)
print("after dummies, shape:", df_features.shape)

before dummies


KeyError: "None of [Index(['A20i', 'B2Cii_Other', 'B3Di_1', 'B3Di_2', 'B3Di_3', 'B3Di_4', 'B3Di_5',\n       'B3Di_6', 'B3Di_7', 'B3Di_8',\n       ...\n       'R2Diiia', 'R2Ei', 'R2Fi', 'R3Dia', 'R3Diia', 'T1_Other_Specify',\n       'U4_Other_specify', 'U19bi', 'U20c_OtherSpecify', 'U20ci'],\n      dtype='object', length=195)] are in the [columns]"

In [ ]:
bool_cols = df_features.select_dtypes(include=['bool', 'uint8']).columns
df_features[bool_cols] = df_features[bool_cols].astype(np.uint8)
print("after cast")

after cast


In [ ]:
print("na count:", df_features.isna().sum().sum())

na count: 0


In [ ]:
df_features = df_features.fillna(df_features.median(numeric_only=True))
print("after fillna")

after fillna


In [ ]:
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df_features.astype(np.float32))

print(f"Features cleaned and scaled. Feature matrix shape: {scaled_features.shape}")


Features cleaned and scaled. Feature matrix shape: (20871, 2868)


In [ ]:
from os import minor
feature_variances = df_features.var()
near_constant = feature_variances[feature_variances < 0.001].index.tolist()
print(f"Near-constant columns: {len(near_constant)}")
dummy_cols = df_features.select_dtypes(include=['uint8', 'bool']).columns
minority_frac = df_features[dummy_cols].mean().apply(lambda p:min(p, 1-p))
rare_dummies = minority_frac[minority_frac < 0.01].index.tolist()
print(f"Dummy columns where minority class is <1% of rows: {len(rare_dummies)}")
dup_mask = df_features.T.duplicated()
duplicated_cols  = df_features.columns[dup_mask].tolist()
print(f"Exact duplicate columns:{len(duplicated_cols)}")
print(duplicated_cols[:20])

Near-constant columns: 1392
Dummy columns where minority class is <1% of rows: 971
Exact duplicate columns:273
['A9i', 'Z1', 'A14_slept', 'Sex', 'NHM', 'secFinancial_Awareness', 'B3D__99_1', 'B3D__8_3', 'B3D__10_3', 'B3D__98_3', 'B3D__99_3', 'B3D__10_9', 'B3D__98_9', 'B3D__99_9', 'D2__99', 'E2__99', 'E4__99', 'E2C__98', 'E2C__99', 'E2H__99']


In [ ]:
corr_matrix = df_features.corr()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row,col] > 0.95
]
print(f"Highly correlated pairs(>0.95):{len(high_corr_pairs)}")
print (high_corr_pairs[:10])

Highly correlated pairs(>0.95):337
[('A07', 'county', np.float64(0.9521966078706605)), ('A14i', 'A18', np.float64(0.9930715613274941)), ('A14iii', 'A19', np.float64(0.9797469237458573)), ('Sex', 'A14ii', np.float64(1.0)), ('Age', 'A18', np.float64(0.9512018208306313)), ('NHM', 'A12ii_length', np.float64(1.0)), ('B3D__98_9', 'B3D__98_1', np.float64(1.0)), ('D1E', 'D1D', np.float64(0.9825363972743051)), ('D1G', 'D1D', np.float64(0.9754020659648512)), ('D1G', 'D1E', np.float64(0.9902961737926945))]


In [ ]:
sparsity = (df_features == 0).sum().sum() / df_features.size
print(f"Overall zero-fraction: {sparsity:.2%}")

Overall zero-fraction: 77.21%


In [ ]:
cols_to_drop = list(set(rare_dummies) | set(duplicated_cols))
print(f"Dropping {len(cols_to_drop)} redundant/near-constant columns")
df_features_clean = df_features.drop(columns=cols_to_drop)
print(f"Reduced from {df_features.shape[1]} to {df_features_clean.shape[1]} columns")
scaled_features = scaler.fit_transform(df_features_clean.astype(np.float32))

Dropping 1105 redundant/near-constant columns
Reduced from 2868 to 1763 columns


In [ ]:
import pandas as pd
data_labeled = pd.read_stata(finaccess, convert_categoricals=False, iterator=True)
variable_labels = data_labeled.variable_labels()
for code in ['A07', 'county', 'A14i', 'A18', 'A14iii', 'A19', 'Sex', 'A14ii', 'Age', 'NHM', 'A12ii_length', 'B3D__98_9', 'B3D__98_1', 'D1E', 'D1D', 'D1G']:
    print(code, '→', variable_labels.get(code, 'no label found'))

A07 → A7. Cluster Number
county → County
A14i → A14i.How old is [NAME] in completed years?
A18 → A18. AGE OF RESPONDENT
A14iii → Relationship of the household member to the head of household
A19 → A19. Relation to HH Head
Sex → Sex of Selected Individual
A14ii → Sex of a household member
Age → Agegroup of Selected Individual
NHM → Number of household members.
A12ii_length → HH_roster_size_ex12ii
B3D__98_9 → B3D__98.9: B3D. Received Payments:Don't know
B3D__98_1 → B3D__98.1: B3D. Received Payments:Don't know
D1E → D1E: Currently saving for retirement or receiving monthly pension
D1D → D1D: Post-Retirement Medical Fund
D1G → D1G: Partial access to Retirement benefits before retirement


In [ ]:
student_related = [c for c in df.columns if 'student' in c.lower() or 'school' in c.lower() or 'educat' in c.lower()]
print(student_related)
print(df['A19'].value_counts() if 'A19' in df.columns else "not found")

['Education', 'saving_for_education']
A19
1    11141
2     5474
3     1539
4     1467
7      965
8      168
6       89
5       28
Name: count, dtype: int64


In [ ]:
import pyreadstat
data_labeled = pd.read_stata(finaccess, convert_categoricals=False, iterator=True)
value_labels = data_labeled.value_labels()
print(data_labeled.variable_labels().get('A19'))
_, meta = pyreadstat.read_dta(finaccess)
print(meta.variable_value_labels.get('A19'))

A19. Relation to HH Head
{1: 'Head of household', 2: 'Spouse', 3: 'Son', 4: 'Daughter', 5: 'Father', 6: 'Mother', 7: 'Other relative', 8: 'Other non-relative e.g domestic workers,friends', 9: "Don't know  (DO NOT READ OUT)", 10: 'Refused to Answer  (DO NOT READ OUT)'}


In [ ]:
_, meta = pyreadstat.read_dta(finaccess)
print(meta.variable_value_labels.get('Education'))
print(meta.variable_value_labels.get('saving_for_education'))
print(meta.variable_labels.get('Education'), '|', meta.variable_labels.get('saving_for_education'))

{1: 'None', 2: 'Primary', 3: 'Secondary', 4: 'Tertiary', 5: 'Other'}
{0: 'no', 1: 'yes'}


AttributeError: 'metadata_container' object has no attribute 'variable_labels'

In [ ]:
student_dependant = df[
    (df['A19'].isin([3, 4])) &                    # Son or Daughter of HH head
    (df['A18'].between(18, 26)) &                  # typical university age range
    (df['Education'] == 4)
]
print(f"Student dependants in sample: {len(student_dependant)} of {len(df)}")

Student dependants in sample: 244 of 20871


In [ ]:
_, meta = pyreadstat.read_dta(finaccess)
labels = meta.column_names_to_labels
for code, label in labels.items():
    if any(term in str(label).lower() for term in ['spending needs', 'spending plans', 'lump', 'allocat', 'emergency' , 'fuliza', 'digital credit', 'overdraft']):
        print(code, '→',label)

B1Ha → B1Ha.  Other Access to emergency
B1J → B1J. Access in Case of Emergency
B1K → B1K. Access in Case of Emergency
C1_13c → C1_13c. LBank/ Micro finance bank Overdraft
C1_16 → C1_16. Loan from mobile provider (e.g Fuliza loan, Equity's Boostika)
D1A__2 → D1A_2: Not contributing - Received lumpsum pension payment
E1iC → E1iC: Number of bank overdraft loans past 12 months
E1iiC → E1iiC: Number of Outstanding bank overdraft loans
E1iiiC → E1iiiC: Reason for taking bank overdraft loan
E1iiiC_Other → E1iiiC: Reason for taking bank overdraft loan product; Others
E1ivC → E1ivC: Collateral used for bank overdraft loan
E1ivC_Other → E1ivC: Collateral used for bank overdraft loan; Others
E1vC → E1vC: Frequency of bank overdraft loan repayments
E1vC_Other → E1vC: Frequency of bank overdraft loan repayments; Others
E1viC → E1viC: Channels used for repaying bank overdraft loan
E1viC_Other → E1viC: Channels used for repaying bank overdraft loan; Others
E1viiC → E1viiC: Amount of bank overdraft lo

In [ ]:
behaviour_cols = {
    'mfhi_22': 'can_access_lumpsum',
    'mfhi_23': 'kept_money_aside_for_emergency',
    'not_difficult': 'emergency_funds_easy_30days',
    'Fuliza_loan': 'used_fuliza_last_12mo',
    'fuliza': 'fuliza_use_any_definition',
    'bank_usage_overall': 'uses_bank_or_mobile_credit',
}
for code in behaviour_cols:
    print(code, '→', meta.variable_value_labels.get(code, 'no labels/ likely continous or alreday binary'))

mfhi_22 → {0: 'No', 1: 'Yes'}
mfhi_23 → {0: 'No', 1: 'Yes'}
not_difficult → {0: 'No', 1: 'Yes'}
Fuliza_loan → {0: 'Non-usage', 1: 'Usage'}
fuliza → {1: 'Currently use', 2: 'Used to use', 3: 'Never used'}
bank_usage_overall → {0: 'Non-usage', 1: 'Usage'}


In [ ]:
# fixing Fuliza cause its not binary
student_dependant = student_dependant.copy()
student_dependant['fuliza_currently_using'] = (student_dependant['fuliza']==1).astype(int)

In [ ]:
behaviour_cols = {
    'mfhi_22': 'can_access_lumpsum',
    'mfhi_23': 'kept_money_aside_for_emergency',
    'not_difficult': 'emergency_funds_easy_30days',
    'Fuliza_loan': 'used_fuliza_last_12mo',
    'fuliza_currently_using': 'currently_uses_fuliza',
    'bank_usage_overall': 'uses_bank_or_mobile_credit',
}
reference_behaviour = student_dependant[list(behaviour_cols.keys())].mean()
reference_behaviour = reference_behaviour.rename(index=behaviour_cols).round(3)
print("Finaccess 2024 - Student DEpendant Finacial Resilience BenchMarks (n=244)")


Finaccess 2024 - Student DEpendant Finacial Resilience BenchMarks (n=244)


In [ ]:
import json
with open('finaccess_student_benchmarks.json', 'w') as f:
    json.dump(reference_behaviour.to_dict(), f, indent=2)

In [ ]:
print(meta.variable_value_labels.get('Fuliza_loan'))
print(student_dependant['Fuliza_loan'].value_counts(dropna=False))
print(student_dependant['fuliza_currently_using'].value_counts(dropna=False))

{0: 'Non-usage', 1: 'Usage'}
Fuliza_loan
0    177
1     67
Name: count, dtype: int64
fuliza_currently_using
0    177
1     67
Name: count, dtype: int64


In [ ]:
behavior_cols = {
    'mfhi_22': 'can_access_lumpsum',
    'mfhi_23': 'kept_money_aside_for_emergency',
    'not_difficult': 'emergency_funds_easy_30days',
    'fuliza_currently_using': 'uses_fuliza',   # single Fuliza indicator, drop the duplicate
    'bank_usage_overall': 'uses_bank_or_mobile_credit',
}

reference_behavior = student_dependant[list(behavior_cols.keys())].mean()
reference_behavior = reference_behavior.rename(index=behavior_cols).round(3)
print(reference_behavior)

with open('/content/drive/MyDrive/finaccess_student_benchmarks.json', 'w') as f:
    json.dump(reference_behavior.to_dict(), f, indent=2)

can_access_lumpsum                0.385
kept_money_aside_for_emergency    0.406
emergency_funds_easy_30days       0.340
uses_fuliza                       0.275
uses_bank_or_mobile_credit        0.791
dtype: float64
